# 50 Startups 多元線性迴歸分析 (CRISP-DM 方法論)
本專案使用 `scikit-learn` 之多元線性迴歸 (Multiple Linear Regression) 演算法，遵循 **CRISP-DM** (跨行業資料採礦標準流程) 解決 50 家新創公司 (50_Startups) 的利潤預測與運營資源配置問題。

### CRISP-DM 六大階段：
1. **商業理解 (Business Understanding)**
2. **資料理解 (Data Understanding)**
3. **資料準備 (Data Preparation)**
4. **模型建構 (Modeling)**
5. **模型評估 (Evaluation)**
6. **模型部署與決策建議 (Deployment)**

## 1. 商業理解 (Business Understanding)
- **商業背景**：創投基金 (VC) 評估 50 家具潛力的新創公司，需要辨識哪些開銷能最有效推動公司獲利 (Profit)。
- **核心目標**：
  1. 建立多元線性迴歸模型預測利潤。
  2. 分析研發 (R&D)、管理 (Administration)、行銷 (Marketing) 與州別 (State) 的回報敏感度。
  3. 提供資料驅動的資本配置建議。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 圖表外觀設定
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 2. 資料理解 (Data Understanding)
載入資料並進行探索性資料分析 (EDA)。

In [ ]:
df = pd.read_csv('50_Startups.csv')
print("Shape:", df.shape)
display(df.head())
print("\n--- 缺失值統計 ---")
print(df.isnull().sum())
print("\n--- 敘述統計 ---")
display(df.describe())

In [ ]:
numeric_cols = ['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']
display(df[numeric_cols].corr())

# 繪製關聯散佈圖
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(df['R&D Spend'], df['Profit'], color='tab:blue')
axes[0].set_title('R&D vs Profit (r=0.973)')
axes[1].scatter(df['Marketing Spend'], df['Profit'], color='tab:green')
axes[1].set_title('Marketing vs Profit (r=0.748)')
axes[2].scatter(df['Administration'], df['Profit'], color='tab:red')
axes[2].set_title('Admin vs Profit (r=0.201)')
plt.tight_layout()
plt.show()

## 3. 資料準備 (Data Preparation)
- 類別變數處理：對 `State` 進行 One-Hot Encoding，設置 `drop_first=True` 避免虛擬變數陷阱 (Dummy Variable Trap)。
- 資料切分：80% 訓練集 (40 筆)、20% 測試集 (10 筆)，`random_state=42`。

In [ ]:
df_encoded = pd.get_dummies(df, columns=['State'], drop_first=True, dtype=int)
X = df_encoded.drop(columns=['Profit'])
y = df_encoded['Profit']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

## 4. 模型建構 (Modeling)
使用 `scikit-learn` 的 `LinearRegression` 擬合多元線性迴歸模型。

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept (beta_0): {model.intercept_:,.2f}")
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.coef_})
display(coef_df)

## 5. 模型評估 (Evaluation)
評估模型在訓練集與測試集之指標 ($R^2$, Adjusted $R^2$, RMSE, MAE) 並進行殘差診斷。

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

r2_test = r2_score(y_test, y_test_pred)
adj_r2_test = 1 - (1 - r2_test) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)

print(f"Test R^2: {r2_test:.4f}")
print(f"Test Adjusted R^2: {adj_r2_test:.4f}")
print(f"Test RMSE: ${rmse_test:,.2f}")
print(f"Test MAE: ${mae_test:,.2f}")

# 繪製預測 vs 實際
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_test_pred, color='navy', edgecolors='k')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('Actual Profit ($)')
plt.ylabel('Predicted Profit ($)')
plt.title(f'Actual vs Predicted Profit (R^2 = {r2_test:.4f})')
plt.show()

## 6. 模型部署與決策建議 (Deployment)
封裝推論函數與提供創投資本配置之具體策略。

In [ ]:
def predict_profit(rd, admin, mkt, state):
    state_fl = 1 if state.title() == 'Florida' else 0
    state_ny = 1 if state.title() == 'New York' else 0
    data = pd.DataFrame([{
        'R&D Spend': rd,
        'Administration': admin,
        'Marketing Spend': mkt,
        'State_Florida': state_fl,
        'State_New York': state_ny
    }])[feature_names]
    return model.predict(data)[0]

sample_profit = predict_profit(160000, 120000, 300000, 'California')
print(f"預測新創公司利潤: ${sample_profit:,.2f}")